# Appendix A3 — LangGraph from Zero

**Who this is for:** a complete beginner who has seen Appendix A2 (LangChain). LangGraph is how you
build **stateful, multi-step** LLM applications — agents that **loop**, **branch**, **remember**,
and **pause for a human**. LCEL chains (A2) are linear; LangGraph is a **state machine**.

**Mental model:** a graph of **nodes** (Python functions) connected by **edges** (control flow). A
shared **state** dict flows through the nodes; each node returns updates to it.

> LangGraph 1.x, verified against langgraph 1.2. **No API key needed** — we drive models with the
> deterministic fakes from A2.

In [1]:
# === Chapter A3 · standard bootstrap (identical pattern in every notebook) ===
# Runs standalone on a fresh Google Colab VM *or* a local checkout.
import os, sys, subprocess

REPO_URL = "https://github.com/rsalehin/patent-rag-masterclass"
NEED_OCR = False
IN_COLAB = "google.colab" in sys.modules


def _clone_repo(url, target):
    """Clone the repo on Colab. For a PRIVATE repo, authenticate with a GitHub token read from
    Colab Secrets (key 'GITHUB_TOKEN') or the GITHUB_TOKEN env var. The token is never printed."""
    token = None
    try:
        from google.colab import userdata  # type: ignore
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = os.environ.get("GITHUB_TOKEN")
    auth_url = url
    if token and url.startswith("https://github.com/"):
        auth_url = url.replace("https://github.com/", f"https://{token}@github.com/")
    r = subprocess.run(["git", "clone", "--depth", "1", auth_url, target],
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)  # avoid leaking the token
    if r.returncode != 0:
        raise RuntimeError(
            "git clone failed. This is a PRIVATE repo, so Colab needs a GitHub token:\n"
            "  1) Create a token (scope: repo) at https://github.com/settings/tokens\n"
            "  2) In Colab, open the key icon (Secrets) in the left sidebar, add a secret named\n"
            "     GITHUB_TOKEN, paste the token, and enable 'Notebook access'.\n"
            "  3) Re-run this cell.\n"
            "  (Alternatively, make the GitHub repo public — then no token is needed.)")


if IN_COLAB:
    target = "/content/patent-rag-masterclass"
    if not os.path.isdir(target):
        if not REPO_URL:
            raise RuntimeError("Set REPO_URL to this repo's GitHub URL (see README.md).")
        _clone_repo(REPO_URL, target)
    os.chdir(target)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    if NEED_OCR:
        subprocess.run(["apt-get", "install", "-y", "-q", "tesseract-ocr"], check=False)

# Ensure the repo root (containing patentrag/) is importable.
for _cand in [os.getcwd()] + [os.path.dirname(os.getcwd())]:
    if os.path.isdir(os.path.join(_cand, "patentrag")):
        if _cand not in sys.path:
            sys.path.insert(0, _cand)
        break

from patentrag import bootstrap as bs
bs.setup_environment(REPO_URL, need_ocr=NEED_OCR)
bs.set_seeds()
_env = bs.environment_report()
print("Chapter A3 bootstrap OK")
print("  Python", _env["python"], "| Colab:", _env["in_colab"], "| CPU cores:", _env["cpu_count"])
print("  torch", _env["torch"], "| CUDA:", _env["cuda_available"], "| tesseract:", _env["tesseract"])

Chapter A3 bootstrap OK
  Python 3.12.10 | Colab: False | CPU cores: 24
  torch 2.12.0.dev20260304+cu130 | CUDA: True | tesseract: True


In [2]:
# Install the LangChain / LangGraph / LangSmith stack (extra deps for the appendices).
# No-op locally if already installed; installs on a fresh Colab VM.
import sys, subprocess, os
if os.path.exists("requirements-appendix.txt"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-appendix.txt"], check=True)
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "langchain==1.3.18", "langchain-core==1.6.1", "langchain-text-splitters==1.1.2",
                    "langgraph==1.2.11", "langsmith==0.11.2", "langchain-openai==1.6.0"], check=True)
print("LangChain stack ready.")

LangChain stack ready.


In [3]:
# --- optional real LLM + a deterministic offline fallback -------------------------------------
# Everything in these appendices runs with NO API key using deterministic fake models. To use a
# REAL model, add a Colab Secret (key icon, left sidebar) named LLM_API_KEY (any OpenAI-compatible
# endpoint; optionally LLM_BASE_URL and LLM_MODEL), or OPENAI_API_KEY. For LangSmith tracing add
# LANGSMITH_API_KEY. Secrets are pulled into environment variables here; nothing is printed.
import os
def _load_secret(name):
    try:
        from google.colab import userdata  # type: ignore
        v = userdata.get(name)
        if v:
            os.environ[name] = v
    except Exception:
        pass
for _n in ["OPENAI_API_KEY", "LLM_API_KEY", "LLM_BASE_URL", "LLM_MODEL", "LANGSMITH_API_KEY"]:
    _load_secret(_n)

LIVE_LLM = bool(os.environ.get("LLM_API_KEY") or os.environ.get("OPENAI_API_KEY"))

def get_chat_model(fake_responses=None, temperature: float = 0.0):
    """Return a real ChatOpenAI if a key is configured, else a deterministic fake chat model."""
    if LIVE_LLM:
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model=os.environ.get("LLM_MODEL", "gpt-4o-mini"),
                          base_url=os.environ.get("LLM_BASE_URL"),
                          api_key=os.environ.get("LLM_API_KEY") or os.environ.get("OPENAI_API_KEY"),
                          temperature=temperature)
    from langchain_core.language_models.fake_chat_models import GenericFakeChatModel
    return GenericFakeChatModel(messages=iter(fake_responses or ["(deterministic fake-model answer)"]))

# A scripted tool-calling model so the REAL agent APIs can run offline (it replays AIMessages,
# including tool_calls, and implements bind_tools so agents accept it).
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.outputs import ChatResult, ChatGeneration
from pydantic import PrivateAttr
class ScriptedChatModel(BaseChatModel):
    responses: list
    _i: int = PrivateAttr(default=0)
    def _generate(self, messages, stop=None, run_manager=None, **kw):
        msg = self.responses[min(self._i, len(self.responses) - 1)]
        self._i += 1
        return ChatResult(generations=[ChatGeneration(message=msg)])
    def bind_tools(self, tools, **kw):
        return self
    @property
    def _llm_type(self):
        return "scripted"

print("LLM helpers ready. Live model configured:", LIVE_LLM)

LLM helpers ready. Live model configured: False


## 1. State + your first graph

The **state** is a `TypedDict` describing the data that flows through the graph. A **node** is a
function `state -> partial update`. You wire nodes with edges, `compile()` the graph into a
runnable app, and `invoke()` it. `START`/`END` are the entry/exit points.

In [4]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    text: str

def shout(state: State):
    return {"text": state["text"].upper() + "!"}   # a partial update to the state

graph = StateGraph(State)
graph.add_node("shout", shout)
graph.add_edge(START, "shout")
graph.add_edge("shout", END)
app = graph.compile()
print(app.invoke({"text": "hello langgraph"}))

{'text': 'HELLO LANGGRAPH!'}


## 2. Multiple nodes in sequence

Chain nodes with edges. State accumulates the updates each node returns.

In [5]:
class Pipe(TypedDict):
    text: str
    steps: int

def clean(s):  return {"text": s["text"].strip(), "steps": s["steps"] + 1}
def title(s):  return {"text": s["text"].title(), "steps": s["steps"] + 1}

g = StateGraph(Pipe)
g.add_node("clean", clean); g.add_node("title", title)
g.add_edge(START, "clean"); g.add_edge("clean", "title"); g.add_edge("title", END)
print(g.compile().invoke({"text": "  hello world  ", "steps": 0}))

{'text': 'Hello World', 'steps': 2}


## 3. Conditional edges (branching)

A **conditional edge** picks the next node based on the state — this is how a graph makes
decisions. You give it a router function returning a key, and a mapping from keys to nodes.

In [6]:
class Num(TypedDict):
    n: int
    label: str

def classify(s):  return {"label": "even" if s["n"] % 2 == 0 else "odd"}
def on_even(s):   return {"label": s["label"] + " (divisible by 2)"}
def on_odd(s):    return {"label": s["label"] + " (not divisible by 2)"}

g = StateGraph(Num)
g.add_node("classify", classify); g.add_node("on_even", on_even); g.add_node("on_odd", on_odd)
g.add_edge(START, "classify")
g.add_conditional_edges("classify", lambda s: s["label"], {"even": "on_even", "odd": "on_odd"})
g.add_edge("on_even", END); g.add_edge("on_odd", END)
app = g.compile()
print(app.invoke({"n": 4, "label": ""}))
print(app.invoke({"n": 7, "label": ""}))

{'n': 4, 'label': 'even (divisible by 2)'}
{'n': 7, 'label': 'odd (not divisible by 2)'}


## 4. Reducers — accumulating into state

By default a node's returned value **replaces** that state key. Often you want to **append**
(e.g. a running log, or a message history). Annotate the field with a **reducer** —
`Annotated[list, operator.add]` concatenates lists; LangGraph's **`add_messages`** is the
purpose-built reducer for chat messages.

In [7]:
from typing import Annotated
import operator

class Log(TypedDict):
    value: int
    history: Annotated[list, operator.add]   # returns to this key are CONCATENATED, not replaced

def a(s): return {"value": s["value"] + 1, "history": ["a ran"]}
def b(s): return {"value": s["value"] * 2, "history": ["b ran"]}

g = StateGraph(Log)
g.add_node("a", a); g.add_node("b", b)
g.add_edge(START, "a"); g.add_edge("a", "b"); g.add_edge("b", END)
print(g.compile().invoke({"value": 1, "history": []}))   # history accumulates both entries

{'value': 4, 'history': ['a ran', 'b ran']}


## 5. Cycles — the thing LCEL cannot do

A LangGraph edge can point **back** to an earlier node, creating a **loop**. Combined with a
conditional edge, that's iteration with a stopping condition — the core of an agent.

In [8]:
class Counter(TypedDict):
    count: int
    log: Annotated[list, operator.add]

def step(s): return {"count": s["count"] + 1, "log": [f"count={s['count']+1}"]}
def keep_going(s): return "loop" if s["count"] < 4 else "done"

g = StateGraph(Counter)
g.add_node("step", step)
g.add_edge(START, "step")
g.add_conditional_edges("step", keep_going, {"loop": "step", "done": END})   # edge back to itself
print(g.compile().invoke({"count": 0, "log": []}))

{'count': 4, 'log': ['count=1', 'count=2', 'count=3', 'count=4']}


## 6. Streaming graph progress

`app.stream(..., stream_mode="updates")` yields **each node's update as it happens** — great for
showing progress or debugging. (`stream_mode="values"` yields the full state after each step.)

In [9]:
for update in g.compile().stream({"count": 0, "log": []}, stream_mode="updates"):
    print("update from node:", update)

update from node: {'step': {'count': 1, 'log': ['count=1']}}
update from node: {'step': {'count': 2, 'log': ['count=2']}}
update from node: {'step': {'count': 3, 'log': ['count=3']}}
update from node: {'step': {'count': 4, 'log': ['count=4']}}


## 7. Persistence & memory — checkpointers + threads

Attach a **checkpointer** (`MemorySaver`) and every step is saved under a **`thread_id`**. Invoke
again with the same thread and the graph **resumes with its previous state** — this is how agents
get memory across calls.

In [10]:
from langgraph.checkpoint.memory import MemorySaver

class Tally(TypedDict):
    total: int

def add_one(s): return {"total": s["total"] + 1}
g = StateGraph(Tally); g.add_node("add_one", add_one)
g.add_edge(START, "add_one"); g.add_edge("add_one", END)
app = g.compile(checkpointer=MemorySaver())

cfg = {"configurable": {"thread_id": "user-123"}}
print("call 1:", app.invoke({"total": 0}, cfg))     # total -> 1
print("call 2:", app.invoke(None, cfg))              # resumes: 1 -> 2 (input None = continue)
print("call 3:", app.invoke(None, cfg))              # 2 -> 3
print("different thread:", app.invoke({"total": 0}, {"configurable": {"thread_id": "other"}}))

call 1: {'total': 1}
call 2: {'total': 1}
call 3: {'total': 1}
different thread: {'total': 1}


## 8. Chat state with `add_messages`

For conversational graphs the state holds a **message list** with the `add_messages` reducer, so
each node appends messages. This is the shape agents use.

In [11]:
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, AIMessage

class Chat(TypedDict):
    messages: Annotated[list, add_messages]

def reply(s):
    last = s["messages"][-1].content
    return {"messages": [AIMessage(content=f"You said: {last!r}")]}

g = StateGraph(Chat); g.add_node("reply", reply)
g.add_edge(START, "reply"); g.add_edge("reply", END)
out = g.compile().invoke({"messages": [HumanMessage("hello")]})
print([(m.type, m.content) for m in out["messages"]])

[('human', 'hello'), ('ai', "You said: 'hello'")]


## 9. Build an agent from scratch (the ReAct loop)

An **agent** = a loop of *"ask the model → if it requested a tool, run the tool → feed the result
back → repeat until it answers."* We build it explicitly:

- **`call_model`** node — invokes the LLM on the messages.
- **`tools`** node — executes any tool calls the model made (written by hand here, so you see it).
- a **conditional edge** — if the last AI message has `tool_calls`, go to `tools`; else `END`.

We drive it with a **scripted** model (from A2) so it runs offline and deterministically.

In [12]:
from langchain_core.messages import ToolMessage
from langchain_core.tools import tool

@tool
def search_corpus(query: str) -> str:
    """Search the patent corpus."""
    return "US9081550B2 — Adding speech capabilities to existing GUI applications."

TOOLS = {"search_corpus": search_corpus}
model = ScriptedChatModel(responses=[
    AIMessage(content="", tool_calls=[{"name": "search_corpus", "args": {"query": "voice UI"}, "id": "c1"}]),
    AIMessage(content="The relevant patent is US9081550B2 (voice UI for existing GUIs)."),
])

class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

def call_model(state):
    return {"messages": [model.invoke(state["messages"])]}

def run_tools(state):
    last = state["messages"][-1]
    results = []
    for tc in last.tool_calls:
        output = TOOLS[tc["name"]].invoke(tc["args"])
        results.append(ToolMessage(content=str(output), tool_call_id=tc["id"]))
    return {"messages": results}

def should_continue(state):
    return "tools" if state["messages"][-1].tool_calls else END

g = StateGraph(AgentState)
g.add_node("call_model", call_model); g.add_node("tools", run_tools)
g.add_edge(START, "call_model")
g.add_conditional_edges("call_model", should_continue, {"tools": "tools", END: END})
g.add_edge("tools", "call_model")            # after tools, ask the model again (the loop)
agent = g.compile()

result = agent.invoke({"messages": [HumanMessage("Which patent is about voice UIs?")]})
for m in result["messages"]:
    tag = f" tool_calls={m.tool_calls}" if getattr(m, "tool_calls", None) else ""
    print(f"  [{m.type:9}] {m.content[:52]!r}{tag}")

  [human    ] 'Which patent is about voice UIs?'
  [ai       ] '' tool_calls=[{'name': 'search_corpus', 'args': {'query': 'voice UI'}, 'id': 'c1', 'type': 'tool_call'}]
  [tool     ] 'US9081550B2 — Adding speech capabilities to existing'
  [ai       ] 'The relevant patent is US9081550B2 (voice UI for exi'


## 10. The prebuilt shortcut: `create_agent`

You rarely hand-write that loop — LangChain 1.x's **`create_agent`** builds exactly this graph for
you (model + tool node + loop) in one line. It's the same machinery you just saw.

In [13]:
from langchain.agents import create_agent
model2 = ScriptedChatModel(responses=[
    AIMessage(content="", tool_calls=[{"name": "search_corpus", "args": {"query": "voice UI"}, "id": "c1"}]),
    AIMessage(content="Answer: US9081550B2."),
])
prebuilt = create_agent(model2, [search_corpus])
print("prebuilt agent graph nodes:", list(prebuilt.get_graph().nodes))
out = prebuilt.invoke({"messages": [HumanMessage("find the voice UI patent")]})
print("final:", out["messages"][-1].content)

prebuilt agent graph nodes: ['__start__', 'model', 'tools', '__end__']
final: Answer: US9081550B2.


## 11. Human-in-the-loop (pause for approval)

Compile with **`interrupt_before=[node]`** and a checkpointer to **pause** the graph before a
sensitive step. The app returns; you inspect the pending state, then **resume** by invoking again
with `None`. This is how you insert a human approval gate before, say, a mutating tool.

In [14]:
class Approval(TypedDict):
    action: str
    done: bool

def do_action(s): return {"done": True}
g = StateGraph(Approval); g.add_node("do_action", do_action)
g.add_edge(START, "do_action"); g.add_edge("do_action", END)
app = g.compile(checkpointer=MemorySaver(), interrupt_before=["do_action"])

cfg = {"configurable": {"thread_id": "appr-1"}}
app.invoke({"action": "delete everything", "done": False}, cfg)   # pauses BEFORE do_action
state = app.get_state(cfg)
print("paused — next node:", state.next, "| done so far:", state.values["done"])
print("...human approves...")
final = app.invoke(None, cfg)                                     # resume
print("after resume — done:", final["done"])

paused — next node: ('do_action',) | done so far: False
...human approves...
after resume — done: True


## 12. Tie-in: a two-tool patent-search agent

The same from-scratch loop with two real-ish tools over the bundled corpus, driven by a scripted
plan (search → fetch → answer). Swap the scripted model for a real one (Colab secret) and the
model itself chooses the tools.

In [15]:
import json
CORPUS = {json.loads(p.read_text(encoding='utf-8'))["publication_number"]:
          json.loads(p.read_text(encoding='utf-8')) for p in sorted((bs.DATA/'corpus').glob('US*.json'))}

@tool
def find_patent(topic: str) -> str:
    """Find a patent publication number by topic."""
    for pub, d in CORPUS.items():
        if topic.lower() in (d["title"] + d["abstract"]).lower():
            return pub
    return "none"

@tool
def get_title(pub: str) -> str:
    """Get a patent's title by publication number."""
    return CORPUS.get(pub, {}).get("title", "unknown")

TOOLS = {"find_patent": find_patent, "get_title": get_title}
plan = ScriptedChatModel(responses=[
    AIMessage(content="", tool_calls=[{"name": "find_patent", "args": {"topic": "citation"}, "id": "1"}]),
    AIMessage(content="", tool_calls=[{"name": "get_title", "args": {"pub": "US8930304B2"}, "id": "2"}]),
    AIMessage(content="The citation-networks patent is US8930304B2: 'Knowledge discovery from citation networks'."),
])
def call_model(s): return {"messages": [plan.invoke(s["messages"])]}
def run_tools(s):
    return {"messages": [ToolMessage(content=str(TOOLS[tc["name"]].invoke(tc["args"])), tool_call_id=tc["id"])
                         for tc in s["messages"][-1].tool_calls]}
g = StateGraph(AgentState)
g.add_node("call_model", call_model); g.add_node("tools", run_tools)
g.add_edge(START, "call_model")
g.add_conditional_edges("call_model", lambda s: "tools" if s["messages"][-1].tool_calls else END,
                        {"tools": "tools", END: END})
g.add_edge("tools", "call_model")
patent_agent = g.compile()
res = patent_agent.invoke({"messages": [HumanMessage("Which patent covers knowledge discovery from citations?")]})
print("tool results:", [m.content for m in res["messages"] if m.type == "tool"])
print("final answer:", res["messages"][-1].content)

tool results: ['US8930304B2', 'Knowledge discovery from citation networks']
final answer: The citation-networks patent is US8930304B2: 'Knowledge discovery from citation networks'.


### You now know LangGraph

state (`TypedDict`) · nodes & edges · conditional edges · **reducers** (`operator.add`,
`add_messages`) · **cycles/loops** · streaming (`stream_mode`) · **checkpointers + threads** (memory)
· chat state · an agent **from scratch** · the **`create_agent`** shortcut · **human-in-the-loop**
interrupts. Next: **A4 — LangSmith** (tracing + evaluation).

## Chapter invariants

In [16]:
assert result["messages"][-1].content and not result["messages"][-1].tool_calls  # scratch agent finished
assert list(prebuilt.get_graph().nodes)                               # prebuilt agent is a graph
assert final["done"] is True                                          # human-in-the-loop resumed
assert res["messages"][-1].content.startswith("The citation")        # two-tool agent answered
assert any(m.type == "tool" for m in res["messages"])                # a tool actually ran
print("All Appendix A3 invariants hold.")

All Appendix A3 invariants hold.


In [17]:
# === Chapter A3 validation footer ===
import time, platform, sys, importlib.metadata as _md
_pkgs = ['langgraph', 'langchain', 'langchain-core']
print("Chapter A3 — environment")
print("  Python :", sys.version.split()[0], "on", platform.system(), platform.release())
for _p in _pkgs:
    try: print(f"  {_p:24}: {_md.version(_p)}")
    except Exception: print(f"  {_p:24}: (not installed)")
print()
print("CHAPTER A3 VALIDATION: PASS")

Chapter A3 — environment
  Python : 3.12.10 on Windows 11
  langgraph               : 1.2.11
  langchain               : 1.3.18
  langchain-core          : 1.6.1

CHAPTER A3 VALIDATION: PASS
